In [1]:
!pip install numpy pandas matplotlib scikit-learn seaborn


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, classification_report, confusion_matrix


In [3]:
!pip install kaggle


In [3]:
from google.colab import files
files.upload()  # Select the kaggle.json file


Saving kaggle.json to kaggle (1).json


{'kaggle (1).json': b'{"username":"ayushkumarr","key":"b8a6af184d1bd22a87f3c5ddecd47ac7"}'}

In [4]:
!pip install -q kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


In [5]:
!kaggle datasets download -d ankushpanday1/lung-cancer-risk-and-prediction-dataset


Dataset URL: https://www.kaggle.com/datasets/ankushpanday1/lung-cancer-risk-and-prediction-dataset
License(s): Community Data License Agreement - Permissive - Version 1.0
lung-cancer-risk-and-prediction-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)


In [6]:
!unzip lung-cancer-risk-and-prediction-dataset.zip

Archive:  lung-cancer-risk-and-prediction-dataset.zip
replace lung_cancer_prediction.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: lung_cancer_prediction.csv  


In [7]:
import pandas as pd

# Load the CSV file (replace with actual CSV filename)
df = pd.read_csv('lung_cancer_prediction.csv')

# Check first few rows and info
print(df.head())
print(df.info())
print(df.isnull().sum())


    Country  Age  Gender Smoking_Status Second_Hand_Smoke  \
0    Russia   82    Male  Former Smoker               Yes   
1  Thailand   66  Female  Former Smoker                No   
2  Colombia   87    Male  Former Smoker                No   
3     Egypt   51  Female  Former Smoker                No   
4  DR Congo   43    Male  Former Smoker                No   

  Air_Pollution_Exposure Occupation_Exposure Rural_or_Urban  \
0                 Medium                  No          Urban   
1                   High                  No          Rural   
2                 Medium                  No          Urban   
3                    Low                 Yes          Rural   
4                   High                  No          Urban   

  Socioeconomic_Status Healthcare_Access  ... Treatment_Access  \
0                 High           Limited  ...          Partial   
1               Middle              Good  ...          Partial   
2                  Low              Poor  ...          P

In [8]:
# Display first few rows
print(df.head())

# Show dataset info (columns, data types, null values)
print(df.info())

# Check for missing values
print(df.isnull().sum())

# Summary statistics
print(df.describe())
print(df.dtypes)



    Country  Age  Gender Smoking_Status Second_Hand_Smoke  \
0    Russia   82    Male  Former Smoker               Yes   
1  Thailand   66  Female  Former Smoker                No   
2  Colombia   87    Male  Former Smoker                No   
3     Egypt   51  Female  Former Smoker                No   
4  DR Congo   43    Male  Former Smoker                No   

  Air_Pollution_Exposure Occupation_Exposure Rural_or_Urban  \
0                 Medium                  No          Urban   
1                   High                  No          Rural   
2                 Medium                  No          Urban   
3                    Low                 Yes          Rural   
4                   High                  No          Urban   

  Socioeconomic_Status Healthcare_Access  ... Treatment_Access  \
0                 High           Limited  ...          Partial   
1               Middle              Good  ...          Partial   
2                  Low              Poor  ...          P

In [9]:
# Check column names
print(df.columns)


Index(['Country', 'Age', 'Gender', 'Smoking_Status', 'Second_Hand_Smoke',
       'Air_Pollution_Exposure', 'Occupation_Exposure', 'Rural_or_Urban',
       'Socioeconomic_Status', 'Healthcare_Access', 'Insurance_Coverage',
       'Screening_Availability', 'Stage_at_Diagnosis', 'Cancer_Type',
       'Mutation_Type', 'Treatment_Access', 'Clinical_Trial_Access',
       'Language_Barrier', 'Mortality_Risk', '5_Year_Survival_Probability',
       'Delay_in_Diagnosis', 'Family_History', 'Indoor_Smoke_Exposure',
       'Tobacco_Marketing_Exposure', 'Final_Prediction'],
      dtype='object')


In [10]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Step 0: Strip column names (remove hidden spaces)
df.columns = df.columns.str.strip()

# Step 1: Fill missing values
if 'Mutation_Type' in df.columns:
    df['Mutation_Type'].fillna('Unknown', inplace=True)
if 'Treatment_Access' in df.columns:
    df['Treatment_Access'].fillna('Unknown', inplace=True)

# Step 2: Remove data leakage features (outcomes known AFTER diagnosis)
leakage_cols = ['Mortality_Risk', '5_Year_Survival_Probability',
                'Stage_at_Diagnosis', 'Cancer_Type', 'Treatment_Access']
existing_leakage_cols = [col for col in leakage_cols if col in df.columns]
df = df.drop(columns=existing_leakage_cols)

# Step 3: Encode categorical variables

# 3a: Binary mapping
binary_map = {'Yes': 1, 'No': 0, 'Male': 1, 'Female': 0, 'Urban': 1, 'Rural': 0}
binary_cols = ['Gender', 'Second_Hand_Smoke', 'Occupation_Exposure',
               'Clinical_Trial_Access', 'Language_Barrier', 'Delay_in_Diagnosis',
               'Family_History', 'Indoor_Smoke_Exposure', 'Tobacco_Marketing_Exposure',
               'Rural_or_Urban']

for col in binary_cols:
    if col in df.columns:
        df[col] = df[col].map(binary_map).fillna(0).astype(int)

# 3b: Ordinal encoding
ordinal_mappings = {
    'Air_Pollution_Exposure': {'Low': 0, 'Medium': 1, 'High': 2},
    'Socioeconomic_Status': {'Low': 0, 'Middle': 1, 'High': 2},
    'Healthcare_Access': {'Poor': 0, 'Limited': 1, 'Good': 2},
    'Insurance_Coverage': {'None': 0, 'Basic': 1, 'Comprehensive': 2},
    'Screening_Availability': {'Low': 0, 'Medium': 1, 'High': 2}
}

for col, mapping in ordinal_mappings.items():
    if col in df.columns:
        df[col] = df[col].map(mapping).fillna(0).astype(int)

# 3c: One-hot encoding for nominal features
nominal_cols = ['Country', 'Smoking_Status']
if 'Mutation_Type' in df.columns:
    nominal_cols.append('Mutation_Type')

existing_nominal_cols = [col for col in nominal_cols if col in df.columns]
df = pd.get_dummies(df, columns=existing_nominal_cols, drop_first=True)

# Step 4: Feature scaling for numerical Age
if 'Age' in df.columns:
    scaler = StandardScaler()
    df['Age_scaled'] = scaler.fit_transform(df[['Age']])
    df = df.drop('Age', axis=1)

# Step 5: Encode target variable if needed
if 'Final_Prediction' in df.columns and df['Final_Prediction'].dtype == 'object':
    df['Final_Prediction'] = df['Final_Prediction'].map({'Yes': 1, 'No': 0})

# Step 6: Final check for NaN in target
if 'Final_Prediction' in df.columns:
    print("NaN in target:", df['Final_Prediction'].isna().sum())


/tmp/ipython-input-2677594618.py:10: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Mutation_Type'].fillna('Unknown', inplace=True)
/tmp/ipython-input-2677594618.py:12: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', t

NaN in target: 0


In [11]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
import pandas as pd

# Separate features and target
X = df.drop('Final_Prediction', axis=1)
y = df['Final_Prediction']

# Drop columns that are completely empty (if any)
X = X.dropna(axis=1, how='all')

# Drop rows where target is missing
nan_rows = y.isnull()
if nan_rows.any():
    X = X[~nan_rows]
    y = y[~nan_rows]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Check missing values before imputation
print("Missing values in X_train before imputation:")
print(X_train.isnull().sum()[X_train.isnull().sum() > 0])

# Mean imputation on numeric features
imputer = SimpleImputer(strategy='mean')
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

# Convert back to DataFrame
X_train_imputed = pd.DataFrame(X_train_imputed, columns=X_train.columns)
X_test_imputed = pd.DataFrame(X_test_imputed, columns=X_test.columns)

# Optional: Check shapes
print("X_train shape:", X_train_imputed.shape)
print("X_test shape:", X_test_imputed.shape)
print("y_train distribution:\n", y_train.value_counts())


Missing values in X_train before imputation:
Series([], dtype: int64)
X_train shape: (368233, 50)
X_test shape: (92059, 50)
y_train distribution:
 Final_Prediction
0    294485
1     73748
Name: count, dtype: int64


In [22]:
    class_proportions = df['Final_Prediction'].value_counts(normalize=True) * 100
    print(class_proportions)

Final_Prediction
0    79.972496
1    20.027504
Name: proportion, dtype: float64


In [12]:
!pip install --upgrade xgboost


In [ ]:
import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import numpy as np
import pandas as pd
from sklearn.utils.class_weight import compute_sample_weight

weights = compute_sample_weight(class_weight='balanced', y=y_train)

# ========================
# 1️⃣ Sample training data for faster tuning
# ========================
sample_size = 100_000
X_train_reset = X_train_imputed.reset_index(drop=True)
y_train_reset = y_train.reset_index(drop=True)

sample_indices = np.random.choice(X_train_reset.shape[0], size=sample_size, replace=False)
X_train_sample = X_train_reset.iloc[sample_indices]
y_train_sample = y_train_reset.iloc[sample_indices]

# ========================
# 2️⃣ Calculate scale_pos_weight for class imbalance
# ========================
neg_count = (y_train_sample == 0).sum()
pos_count = (y_train_sample == 1).sum()
scale_pos_weight = neg_count / pos_count
print(f"Scale_pos_weight for XGBoost: {scale_pos_weight:.2f}")

# ========================
# 3️⃣ Define XGBoost classifier for tuning
# ========================
xgb_clf = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='auc',
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1
)

# ========================
# 4️⃣ Hyperparameter search space
# ========================
param_dist = {
    'n_estimators': [100, 150, 200],
    'max_depth': [4, 5, 6, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9],
    'gamma': [0, 0.1, 0.2],
    'reg_alpha': [0, 0.01, 0.1],
    'reg_lambda': [1, 1.5, 2]
}

# ========================
# 5️⃣ RandomizedSearchCV for hyperparameter tuning
# ========================
rand_search = RandomizedSearchCV(
    estimator=xgb_clf,
    param_distributions=param_dist,
    n_iter=20,
    scoring='roc_auc',
    cv=2,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

# ========================
# 6️⃣ Fit on sampled data
# ========================
rand_search.fit(X_train_sample, y_train_sample)
print("Best hyperparameters:", rand_search.best_params_)

# ========================
# 7️⃣ Train final model on full training data (no early stopping)
# ========================
final_params = {k: v for k, v in rand_search.best_params_.items() if k != 'n_estimators'}

final_xgb = xgb.XGBClassifier(
    **final_params,
    objective='binary:logistic',
    eval_metric='auc',
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    n_estimators=500  # use more trees since no early stopping
)

final_xgb.fit(X_train_imputed, y_train,sample_weight=weights)

# ========================
# 8️⃣ Predict and evaluate
# ========================
y_pred = final_xgb.predict(X_test_imputed)
y_pred_proba = final_xgb.predict_proba(X_test_imputed)[:, 1]

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f"ROC-AUC Score: {roc_auc:.4f}")

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)


Scale_pos_weight for XGBoost: 4.02
Fitting 2 folds for each of 20 candidates, totalling 40 fits


In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from imblearn.pipeline import Pipeline

estimators = [
    ('rf', RandomForestClassifier(random_state=42)),
    ('gb', GradientBoostingClassifier(random_state=42)),
    ('lr', LogisticRegression(max_iter=1000, random_state=42))
]

pipeline = Pipeline([
    ('voting', VotingClassifier(
        estimators=estimators,
        voting='soft',
        weights=[2, 1, 1],
        n_jobs=-1
    ))
])

param_grid = {
    'voting__rf__n_estimators': [100, 200],
    'voting__rf__max_depth': [None, 10],
    'voting__gb__learning_rate': [0.05, 0.1],
    'voting__gb__n_estimators': [100, 200],
    'voting__lr__C': [0.1, 1.0, 10.0]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=cv,
    n_jobs=-1,
    verbose=2
)

# Fit using balanced training data with SMOTE already applied
grid_search.fit(X_train_balanced, y_train_balanced)

print("Best parameters:", grid_search.best_params_)
print("Best CV AUC-ROC:", grid_search.best_score_)

# Evaluate on test set
from sklearn.metrics import classification_report, roc_auc_score

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test_imputed)
y_proba = best_model.predict_proba(X_test_imputed)[:, 1]

print(classification_report(y_test, y_pred))
print("Test ROC-AUC:", roc_auc_score(y_test, y_proba))


Fitting 5 folds for each of 48 candidates, totalling 240 fits
